# Phase 3 V1 safe recovery and controlled fine-tuning

This run starts from the Phase 3 context BiLSTM checkpoint behind the user's earlier 0.4311 Codabench result. It does not restart Phase 1 training.


## What this version changes

The previous 0.4311 submission remains the incumbent. The earlier 0.3941 run used a newly initialized Phase 3 sequence head, so this notebook restores the already-trained context BiLSTM and tests only a short, low-rate continuation from those saved weights.

The continuation trains on the labeled training documents only. Development documents decide whether it is worth promoting. A candidate is accepted only if mean fixed-13 CL/IT development Macro-F1 improves by at least 0.01 and neither domain falls by more than 0.01. Otherwise, the notebook packages the exact saved prediction file from the 0.4311 run. This guard reduces the chance of replacing the known better result; it cannot guarantee a higher hidden-test score.

For Kaggle, keep the existing Phase 1 and Phase 3 data inputs attached, and add these two supplied files as an input dataset: `best_phase3_v1_context_bilstm.pt` and `best_04311_prediction.json`. The notebook checks both before it starts. The V1 backbone is loaded and frozen only to extract sentence features. It is never pretrained again.


In [ ]:
import os, json, random, time, copy, gc, hashlib, zipfile
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import nullcontext

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("/kaggle/working/phase3_rr_v1_safe_retrain_v5")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SENTENCE_TOKENS = 512
FEATURE_BATCH_SIZE = 4
USE_MIXED_PRECISION = True

print("Device:", DEVICE)
if DEVICE.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))
print("Output directory:", OUTPUT_DIR)


In [ ]:
# Find each supplied asset under Kaggle's mounted input datasets.
INPUT_ROOT = Path("/kaggle/input")
ASSET_OVERRIDES = {
    # Optional explicit paths if Kaggle has more than one file with the same name.
    # Example: "v1_model.pt": "/kaggle/input/my-model/v1_model.pt"
}
REQUIRED_FILES = [
    "bpe_en_25k.json", "v1_model.pt", "label_vocab.json", "CL_train.json", "CL_dev.json",
    "IT_train.json", "IT_dev.json", "CL_test.json", "IT_test.json", "best_phase3_v1_context_bilstm.pt", "best_04311_prediction.json"
]

def resolve_input_file(filename):
    override = ASSET_OVERRIDES.get(filename)
    if override:
        path = Path(override)
        if not path.is_file(): raise FileNotFoundError(f"Configured asset does not exist: {path}")
        return path
    if not INPUT_ROOT.exists():
        raise FileNotFoundError("/kaggle/input is missing. Run this notebook in Kaggle and attach the project input datasets.")
    hits = sorted(p for p in INPUT_ROOT.rglob(filename) if p.is_file())
    if not hits: raise FileNotFoundError(f"Required file {filename!r} was not found under {INPUT_ROOT}.")
    if len(hits) > 1:
        raise RuntimeError(f"Found multiple {filename!r} files. Set ASSET_OVERRIDES[{filename!r}] to the intended Kaggle path: {hits}")
    return hits[0]

ASSETS = {name: resolve_input_file(name) for name in REQUIRED_FILES}
for name, path in ASSETS.items(): print(f"{name}: {path}")


## Original V1 tokenizer loading

The following tokenizer loading and tokenizer implementation cells are copied verbatim from original V1 notebook cells 13 and 14. The original tokenizer uses the `itos`/`merges` payload and word-by-word byte-level BPE.

In [ ]:
#tokenizer from A1
import os
import json

BPE_FILENAME = "bpe_en_25k.json"
bpe_candidates = []

for root, dirs, files in os.walk("/kaggle/input"):
    if BPE_FILENAME in files:
        bpe_candidates.append(os.path.join(root, BPE_FILENAME))

if len(bpe_candidates) == 0:
    raise FileNotFoundError(
        f"{BPE_FILENAME} was not found inside /kaggle/input/. "
        "Make sure the BPE-25k dataset is attached."
    )

BPE_PATH = bpe_candidates[0]

print("Tokenizer found:", BPE_PATH)


# Load the vocabulary and learned BPE merges.
with open(BPE_PATH, "r", encoding="utf-8") as f:
    bpe_data = json.load(f)

bpe_itos = bpe_data["itos"]
bpe_merges = bpe_data["merges"]
bpe_vocab_size = bpe_data["vocab_size"]

# Make sure we have loaded the tokenizer we actually want.
assert bpe_vocab_size == 25_000
assert len(bpe_itos) == 25_000

print("Vocabulary size:", bpe_vocab_size)
print("Number of merge rules:", len(bpe_merges))


# Token -> ID mapping
bpe_stoi = {
    token: idx
    for idx, token in enumerate(bpe_itos)
}
merge_to_id = {}

for merge_index, merge in enumerate(bpe_merges):
    left_token, right_token = merge

    left_id = bpe_stoi[left_token]
    right_id = bpe_stoi[right_token]

    new_token_id = 261 + merge_index
    merge_to_id[(left_id, right_id)] = new_token_id

PAD_TOKEN_ID = bpe_stoi["<pad>"]
UNK_TOKEN_ID = bpe_stoi["<unk>"]
BOS_TOKEN_ID = bpe_stoi["<s>"]
EOS_TOKEN_ID = bpe_stoi["</s>"]
WORD_BOUNDARY_ID = bpe_stoi["<wb>"]

print("\nSpecial token IDs:")
print("PAD:", PAD_TOKEN_ID)
print("UNK:", UNK_TOKEN_ID)
print("BOS:", BOS_TOKEN_ID)
print("EOS:", EOS_TOKEN_ID)
print("WORD BOUNDARY:", WORD_BOUNDARY_ID)

In [ ]:
#Tokenizer Sanity Check
#Before using the tokenizer on the full legal corpus,let's check a small example and make sure encoding and decoding work correctly.

class ByteLevelBPE25k:
    def __init__(self, itos, stoi, merge_to_id, merges):
        self.itos = itos
        self.stoi = stoi
        self.merge_to_id = merge_to_id
        self.merges = merges
        self.vocab_size = len(itos)

        self.pad_token_id = stoi["<pad>"]
        self.unk_token_id = stoi["<unk>"]
        self.bos_token_id = stoi["<s>"]
        self.eos_token_id = stoi["</s>"]
        self.word_boundary_id = stoi["<wb>"]

    def _encode_word(self, word):
        # Start with the UTF-8 bytes of the word.
        sequence = [
            4 + byte
            for byte in word.encode("utf-8")
        ]

        # Our BPE representation keeps a word-boundary token.
        sequence.append(self.word_boundary_id)

        while len(sequence) > 1:
            best_position = None
            best_token_id = None

            for i in range(len(sequence) - 1):
                pair = (sequence[i], sequence[i + 1])
                new_token_id = self.merge_to_id.get(pair)

                if new_token_id is None:
                    continue

                if (
                    best_token_id is None
                    or new_token_id < best_token_id
                ):
                    best_token_id = new_token_id
                    best_position = i

            if best_position is None:
                break

            i = best_position
            sequence = (
                sequence[:i]
                + [best_token_id]
                + sequence[i + 2:]
            )

        return sequence

    def encode(self, text, add_bos=False, add_eos=False):
        # The original tokenizer was trained word-by-word.
        words = text.split()

        token_ids = []

        if add_bos:
            token_ids.append(self.bos_token_id)

        for word in words:
            token_ids.extend(
                self._encode_word(word)
            )

        if add_eos:
            token_ids.append(self.eos_token_id)

        return token_ids

    def _token_to_bytes(self, token_id):
        token = self.itos[token_id]

        # Raw byte token.
        if token.startswith("<0x") and token.endswith(">"):
            return bytes([int(token[3:-1], 16)])

        if token == "<wb>":
            return b" "

        if token in {"<pad>", "<unk>", "<s>", "</s>"}:
            return b""

        # Learned BPE token.
        if token.startswith("<bpe_"):
            merge_index = token_id - 261

            if 0 <= merge_index < len(self.merges):
                left_token, right_token = self.merges[merge_index]

                left_id = self.stoi[left_token]
                right_id = self.stoi[right_token]

                return (
                    self._token_to_bytes(left_id)
                    + self._token_to_bytes(right_id)
                )

        return b""

    def decode(self, token_ids):
        output = bytearray()

        for token_id in token_ids:
            if not 0 <= int(token_id) < self.vocab_size:
                continue

            output.extend(
                self._token_to_bytes(int(token_id))
            )

        return bytes(output).decode(
            "utf-8",
            errors="replace"
        )


# Creating the tokenizer.
tokenizer = ByteLevelBPE25k(
    itos=bpe_itos,
    stoi=bpe_stoi,
    merge_to_id=merge_to_id,
    merges=bpe_merges
)


# Test it on a small legal-style sentence.
sample_text = (
    "The Supreme Court of India has jurisdiction "
    "over constitutional matters."
)

sample_tokens = tokenizer.encode(sample_text)
decoded_text = tokenizer.decode(sample_tokens)

print("Original text:")
print(sample_text)
print("\nToken IDs:")
print(sample_tokens)
print("\nNumber of tokens:", len(sample_tokens))
print("\nDecoded text:")
print(decoded_text)
print("\nVocabulary size:", tokenizer.vocab_size)
assert tokenizer.vocab_size == 25_000
assert len(sample_tokens) > 0
print("\nTokenizer sanity check passed.")

In [ ]:
# Load and audit the Phase 3 label vocabulary and data schemas.
LABEL_VOCAB_PATH = ASSETS["label_vocab.json"]
with open(LABEL_VOCAB_PATH, "r", encoding="utf-8") as f: label_to_id = json.load(f)
label_to_id = {str(k): int(v) for k, v in label_to_id.items()}
assert len(label_to_id) == 13, f"Expected 13 labels, found {len(label_to_id)}"
assert sorted(label_to_id.values()) == list(range(13)), "label_vocab.json indices must be contiguous 0..12"
id_to_label = {i: label for label, i in label_to_id.items()}
label_set = set(label_to_id)
print("Official label mapping:", label_to_id)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f: return json.load(f)

def audit_labeled(path, split_name):
    records = load_json(path)
    if not isinstance(records, list): raise TypeError(f"{split_name}: expected a list of documents")
    ids=[]; sentence_count=0; empty_count=0; counts=Counter()
    for row_num, row in enumerate(records):
        if not isinstance(row, dict): raise TypeError(f"{split_name} row {row_num}: expected object")
        for key in ("id", "text", "labels"):
            if key not in row: raise ValueError(f"{split_name} row {row_num}: missing {key!r}")
        if not isinstance(row["text"], list) or not isinstance(row["labels"], list):
            raise TypeError(f"{split_name} row {row_num}: text and labels must be lists")
        if len(row["text"]) == 0: raise ValueError(f"{split_name} {row['id']}: empty document")
        if len(row["text"]) != len(row["labels"]):
            raise ValueError(f"{split_name} {row['id']}: {len(row['text'])} sentences but {len(row['labels'])} labels")
        ids.append(str(row["id"])); sentence_count += len(row["text"])
        empty_count += sum(not str(x).strip() for x in row["text"])
        unknown=set(row["labels"])-label_set
        if unknown: raise ValueError(f"{split_name} {row['id']}: unknown labels {sorted(unknown)}")
        counts.update(row["labels"])
    if len(ids) != len(set(ids)): raise ValueError(f"{split_name}: duplicate document IDs")
    print(f"{split_name}: {len(records)} documents, {sentence_count} sentences, {empty_count} empty sentences")
    if counts: print("  label counts:", dict(counts))
    return records, set(ids)

CL_TRAIN, cl_train_ids = audit_labeled(ASSETS["CL_train.json"], "CL train")
CL_DEV, cl_dev_ids = audit_labeled(ASSETS["CL_dev.json"], "CL dev")
IT_TRAIN, it_train_ids = audit_labeled(ASSETS["IT_train.json"], "IT train")
IT_DEV, it_dev_ids = audit_labeled(ASSETS["IT_dev.json"], "IT dev")
assert not (cl_train_ids & cl_dev_ids), "CL train/dev document leakage"
assert not (it_train_ids & it_dev_ids), "IT train/dev document leakage"

def document_fingerprints(records):
    return {hashlib.sha256(json.dumps([str(s) for s in row["text"]],ensure_ascii=False,
        separators=(",",":")).encode("utf-8")).hexdigest():str(row["id"]) for row in records}
named_splits={"CL_train":CL_TRAIN,"CL_dev":CL_DEV,"IT_train":IT_TRAIN,"IT_dev":IT_DEV}
fingerprints={name:document_fingerprints(rows) for name,rows in named_splits.items()}
for i,left in enumerate(named_splits):
    for right in list(named_splits)[i+1:]:
        overlap=set(fingerprints[left]) & set(fingerprints[right])
        if overlap: raise ValueError(f"Exact document-text overlap between {left} and {right}: {len(overlap)}")
print("No exact document-text duplicates across the four labeled train/dev splits.")

def load_test_docs(path, domain):
    data = load_json(path)
    if not isinstance(data, dict): raise TypeError(f"{domain} test: expected mapping document ID -> sentence list")
    docs=[]
    for doc_id, sentences in data.items():
        if not isinstance(sentences, list): raise TypeError(f"{domain} test {doc_id}: expected sentence list")
        docs.append((str(doc_id), [str(s) for s in sentences]))
    print(f"{domain} test: {len(docs)} documents, {sum(len(x[1]) for x in docs)} sentences")
    return docs

CL_TEST = load_test_docs(ASSETS["CL_test.json"], "CL")
IT_TEST = load_test_docs(ASSETS["IT_test.json"], "IT")
for domain,docs in (("CL",CL_TEST),("IT",IT_TEST)):
    empty=[doc_id for doc_id,sentences in docs if not sentences]
    if empty: raise ValueError(f"{domain} test contains empty documents: {empty[:5]}")


In [ ]:
# Match V1 pretraining convention: no BOS/EOS, and cap long sentences consistently.
def encode_sentence(sentence):
    ids = tokenizer.encode(str(sentence), add_bos=False, add_eos=False)
    if not ids: ids = [UNK_TOKEN_ID]
    truncated = len(ids) > MAX_SENTENCE_TOKENS
    if truncated: ids = ids[:MAX_SENTENCE_TOKENS]
    return ids, truncated

## Original V1 architecture

This is original V1 notebook cell 26 copied verbatim. It defines the pretrained V1 module names and parameter shapes. The following Phase 3 cells instantiate this exact class from the checkpoint configuration and load its saved weights strictly.

In [ ]:
# 9. V1 Decoder-Only Transformer
# This is our first architecture built from scratch.
# V1 uses:
# - RMSNorm
# - RoPE
# - Grouped Query Attention
# - Causal self-attention
# - Standard GELU FFN
# - Residual connections
# - KV cache for autoregressive generation

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------
# RMSNorm
# ---------------------------------------------------------

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(
            dim=-1,
            keepdim=True
        )

        x = x * torch.rsqrt(
            rms + self.eps
        )

        return self.weight * x


# ---------------------------------------------------------
# Rotary Positional Embeddings
# ---------------------------------------------------------

class RotaryEmbedding(nn.Module):
    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=10000
    ):
        super().__init__()

        self.head_dim = head_dim
        self.max_seq_len = max_seq_len

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2
                ).float()
                / head_dim
            )
        )

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False
        )

        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        positions = torch.arange(
            seq_len,
            device=self.inv_freq.device
        ).float()

        freqs = torch.outer(
            positions,
            self.inv_freq
        )

        self.register_buffer(
            "cos_cached",
            freqs.cos(),
            persistent=False
        )

        self.register_buffer(
            "sin_cached",
            freqs.sin(),
            persistent=False
        )

    def forward(
        self,
        q,
        k,
        position_offset=0
    ):
        seq_len = q.size(-2)

        required_len = (
            position_offset + seq_len
        )

        if required_len > self.max_seq_len:
            self._build_cache(required_len)
            self.max_seq_len = required_len

        cos = self.cos_cached[
            position_offset:
            position_offset + seq_len
        ]

        sin = self.sin_cached[
            position_offset:
            position_offset + seq_len
        ]

        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)

        q = self._apply_rotary(
            q,
            cos,
            sin
        )

        k = self._apply_rotary(
            k,
            cos,
            sin
        )

        return q, k

    @staticmethod
    def _apply_rotary(x, cos, sin):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]

        rotated = torch.stack(
            (-x2, x1),
            dim=-1
        ).flatten(-2)

        cos_full = torch.repeat_interleave(
            cos,
            2,
            dim=-1
        )

        sin_full = torch.repeat_interleave(
            sin,
            2,
            dim=-1
        )

        return (
            x * cos_full
            + rotated * sin_full
        )


# ---------------------------------------------------------
# Grouped Query Attention
# ---------------------------------------------------------

class GQAAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0

        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head

        self.kv_repeat = (
            n_head // n_kv_head
        )

        self.q_proj = nn.Linear(
            n_embd,
            n_head * self.head_dim,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.rope = RotaryEmbedding(
            head_dim=self.head_dim,
            max_seq_len=max_seq_len
        )

        self.dropout = dropout

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.n_head,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        past_len = 0

        if past_key_value is not None:
            past_k, past_v = past_key_value
            past_len = past_k.size(-2)

        # Apply RoPE to the new Q and K positions.
        q, k = self.rope(
            q,
            k,
            position_offset=past_len
        )

        # Add the new K/V to the cached K/V during generation.
        if past_key_value is not None:
            k = torch.cat(
                [past_k, k],
                dim=-2
            )

            v = torch.cat(
                [past_v, v],
                dim=-2
            )

        present = None

        if use_cache:
            present = (k, v)

        # GQA: each KV head is shared by a group of query heads.
        if self.kv_repeat > 1:
            k = k.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

            v = v.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

        # During training, each token can only attend to
        # itself and tokens before it.
        #
        # IMPORTANT:
        # For scaled_dot_product_attention, True means
        # "this position is allowed to attend".
        if past_key_value is None:
            attn_mask = torch.tril(
                torch.ones(
                    T,
                    T,
                    device=x.device,
                    dtype=torch.bool
                )
            )
        else:
            # During cached decoding, all cached positions
            # are valid keys for the current token.
            attn_mask = None

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        y = y.transpose(
            1,
            2
        ).contiguous()

        y = y.view(
            B,
            T,
            C
        )

        y = self.out_proj(y)

        return y, present


# ---------------------------------------------------------
# Standard FFN
# ---------------------------------------------------------

class FeedForward(nn.Module):
    def __init__(
        self,
        n_embd,
        multiplier=4,
        dropout=0.0
    ):
        super().__init__()

        hidden_dim = int(
            n_embd * multiplier
        )

        self.fc1 = nn.Linear(
            n_embd,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_embd
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)

        return x


# ---------------------------------------------------------
# Transformer Block
# ---------------------------------------------------------

class TransformerBlock(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.norm1 = RMSNorm(n_embd)

        self.attention = GQAAttention(
            n_embd=n_embd,
            n_head=n_head,
            n_kv_head=n_kv_head,
            max_seq_len=max_seq_len,
            dropout=dropout
        )

        self.norm2 = RMSNorm(n_embd)

        self.ffn = FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        # Pre-norm attention followed by a residual connection.
        attn_output, present = self.attention(
            self.norm1(x),
            past_key_value=past_key_value,
            use_cache=use_cache
        )

        x = x + attn_output

        # Pre-norm FFN followed by a residual connection.
        x = x + self.ffn(
            self.norm2(x)
        )

        return x, present


# ---------------------------------------------------------
# V1 Language Model
# ---------------------------------------------------------

class V1LegalLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embd,
        n_layer,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.n_embd = n_embd

        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                n_embd=n_embd,
                n_head=n_head,
                n_kv_head=n_kv_head,
                max_seq_len=max_seq_len,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout
            )
            for _ in range(n_layer)
        ])

        self.final_norm = RMSNorm(
            n_embd
        )

        # Tie the LM head to the token embedding weights.
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False
    ):
        x = self.token_embedding(
            input_ids
        )

        presents = [] if use_cache else None

        for layer_index, block in enumerate(
            self.blocks
        ):
            past = None

            if past_key_values is not None:
                past = past_key_values[
                    layer_index
                ]

            x, present = block(
                x,
                past_key_value=past,
                use_cache=use_cache
            )

            if use_cache:
                presents.append(
                    present
                )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits, presents

In [ ]:
# Load the already-trained V1 checkpoint. This does not execute Phase 1 training.
CHECKPOINT_PATH = resolve_input_file("v1_model.pt")
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
if not isinstance(checkpoint, dict) or "model_state_dict" not in checkpoint:
    raise ValueError("Expected the supplied V1 checkpoint wrapper with model_state_dict")
model_config = checkpoint.get("config", {})
expected_config = {"vocab_size":25000,"n_embd":768,"n_layer":12,"n_head":12,
                   "n_kv_head":4,"ffn_multiplier":4,"context_length":1024}
for key, expected in expected_config.items():
    if key in model_config and int(model_config[key]) != expected:
        raise ValueError(f"Checkpoint config mismatch for {key}: {model_config[key]} vs {expected}")

backbone = V1LegalLM(
    vocab_size=int(model_config.get("vocab_size",25000)),
    n_embd=int(model_config.get("n_embd",768)),
    n_layer=int(model_config.get("n_layer",12)),
    n_head=int(model_config.get("n_head",12)),
    n_kv_head=int(model_config.get("n_kv_head",4)),
    max_seq_len=int(model_config.get("context_length",1024)),
    ffn_multiplier=int(model_config.get("ffn_multiplier",4)),
    dropout=float(model_config.get("dropout",0.0)))
load_result = backbone.load_state_dict(checkpoint["model_state_dict"], strict=True)
assert not load_result.missing_keys and not load_result.unexpected_keys
backbone = backbone.to(DEVICE)
print("Strict V1 checkpoint load succeeded.")
print("Checkpoint:", CHECKPOINT_PATH)
print("Best Phase 1 validation loss:", checkpoint.get("best_val_loss"))
print("Backbone parameters:", f"{sum(p.numel() for p in backbone.parameters()):,}")

In [ ]:
# Reuse the V1 token embedding, Transformer blocks, and final RMSNorm.
# This is the same hidden-state path used inside V1LegalLM.forward, omitting only
# the 25,000-way language-model projection. Verify that equivalence below.
def encode_v1_hidden(v1_model, input_ids):
    x = v1_model.token_embedding(input_ids)
    for block in v1_model.blocks:
        x, _ = block(x, past_key_value=None, use_cache=False)
    return v1_model.final_norm(x)

probe_ids=torch.tensor([[4,5,6,7,8]],dtype=torch.long,device=DEVICE)
backbone.eval()
with torch.no_grad():
    full_logits,_=backbone(probe_ids,use_cache=False)
    hidden=encode_v1_hidden(backbone,probe_ids)
    projected_logits=backbone.lm_head(hidden)
assert torch.allclose(full_logits,projected_logits,atol=1e-5,rtol=1e-5)
print("V1 hidden extraction matches original forward path; max difference:",
      float((full_logits-projected_logits).abs().max()))

print("V1 is loaded for frozen feature extraction only; no Phase 1 or sentence-head training starts here.")


In [ ]:
# Fixed-label metric: all 13 official roles count, including unsupported labels.
N_LABELS = len(label_to_id)
def metrics_from_ids(gold, pred):
    if len(gold) != len(pred) or not gold:
        raise ValueError("Metric inputs must be non-empty and have equal lengths")
    cm = np.zeros((N_LABELS, N_LABELS), dtype=np.int64)
    for y, p in zip(gold, pred): cm[int(y), int(p)] += 1
    tp = np.diag(cm).astype(float)
    support = cm.sum(axis=1).astype(float)
    predicted = cm.sum(axis=0).astype(float)
    precision = np.divide(tp, predicted, out=np.zeros_like(tp), where=predicted > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    f1 = np.divide(2*precision*recall, precision+recall,
                   out=np.zeros_like(tp), where=(precision+recall) > 0)
    active = support > 0
    per_class = {id_to_label[i]: {
        "precision": float(precision[i]), "recall": float(recall[i]),
        "f1": float(f1[i]), "support": int(support[i]),
        "predicted_count": int(predicted[i])} for i in range(N_LABELS)}
    return {"macro_precision_all_13": float(precision.mean()),
            "macro_recall_all_13": float(recall.mean()),
            "macro_f1": float(f1.mean()),
            "macro_f1_gold_supported_diagnostic": float(f1[active].mean()) if active.any() else 0.0,
            "gold_supported_label_count": int(active.sum()),
            "per_class": per_class, "confusion_matrix": cm.tolist()}

## Frozen V1 feature extraction

The notebook extracts final-token, mean, and max views from the already-trained V1 hidden states. The recovered context model uses the final-token representation, matching the earlier 0.4311 model. V1 stays frozen; only its Phase 3 context head is considered for continuation.


In [ ]:
backbone = backbone.to(DEVICE).eval()
for parameter in backbone.parameters(): parameter.requires_grad_(False)

def make_document(record, domain, split, test=False):
    if test:
        doc_id, sentences = record
        labels = None
    else:
        doc_id, sentences = str(record["id"]), record["text"]
        labels = [label_to_id[x] for x in record["labels"]]
        if len(sentences) != len(labels):
            raise ValueError(f"{domain}/{split}/{doc_id}: sentence/label alignment mismatch")
    if not sentences: raise ValueError(f"{domain}/{split}/{doc_id}: empty document")
    return {"key": f"{domain}::{doc_id}", "id": str(doc_id), "domain": domain,
            "split": split, "sentences": [str(x) for x in sentences], "labels": labels}

DOCUMENTS = []
for domain, split_name, rows in [
    ("CL", "train", CL_TRAIN), ("CL", "dev", CL_DEV),
    ("IT", "train", IT_TRAIN), ("IT", "dev", IT_DEV)]:
    DOCUMENTS.extend(make_document(row, domain, split_name) for row in rows)
for domain, rows in [("CL", CL_TEST), ("IT", IT_TEST)]:
    DOCUMENTS.extend(make_document(row, domain, "test", test=True) for row in rows)

all_keys = [row["key"] for row in DOCUMENTS]
if len(all_keys) != len(set(all_keys)): raise ValueError("Duplicate domain/document key")
test_ids = [row["id"] for row in DOCUMENTS if row["split"] == "test"]
if len(test_ids) != len(set(test_ids)):
    raise ValueError("Test document IDs must be unique across CL and IT for flat Codabench JSON")

def extract_document_features(documents):
    examples = []
    total_truncated = 0
    for doc in documents:
        for sentence_index, sentence in enumerate(doc["sentences"]):
            ids, truncated = encode_sentence(sentence)
            total_truncated += int(truncated)
            examples.append((doc["key"], sentence_index, ids))
    grouped = defaultdict(list)
    for start in range(0, len(examples), FEATURE_BATCH_SIZE):
        batch = examples[start:start+FEATURE_BATCH_SIZE]
        seqs = [torch.tensor(row[2], dtype=torch.long) for row in batch]
        lengths = torch.tensor([len(x) for x in seqs], dtype=torch.long, device=DEVICE)
        input_ids = pad_sequence(seqs, batch_first=True, padding_value=PAD_TOKEN_ID).to(DEVICE)
        with torch.inference_mode():
            if DEVICE.type == "cuda" and USE_MIXED_PRECISION:
                dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
                with torch.autocast(device_type="cuda", dtype=dtype):
                    hidden = encode_v1_hidden(backbone, input_ids)
            else:
                hidden = encode_v1_hidden(backbone, input_ids)
            hidden = hidden.float()
            rows = torch.arange(input_ids.shape[0], device=DEVICE)
            last = hidden[rows, lengths-1]
            mask = torch.arange(hidden.shape[1], device=DEVICE)[None, :] < lengths[:, None]
            mean = (hidden * mask.unsqueeze(-1)).sum(dim=1) / lengths.unsqueeze(-1)
            maximum = hidden.masked_fill(~mask.unsqueeze(-1), -torch.inf).amax(dim=1)
            multi = torch.cat([last, mean, maximum], dim=-1)
        for j, (key, sentence_index, _) in enumerate(batch):
            grouped[key].append((sentence_index, last[j].cpu(), multi[j].cpu()))

    result = {}
    for doc in documents:
        rows = sorted(grouped[doc["key"]], key=lambda x: x[0])
        if len(rows) != len(doc["sentences"]):
            raise RuntimeError(f"{doc['key']}: feature count does not match sentence count")
        item = dict(doc)
        item["features_last"] = torch.stack([x[1] for x in rows]).contiguous()
        item["features_multi"] = torch.stack([x[2] for x in rows]).contiguous()
        if doc["labels"] is not None:
            item["labels_tensor"] = torch.tensor(doc["labels"], dtype=torch.long)
        result[doc["key"]] = item
    print(f"Frozen V1 features: {len(result)} documents, {len(examples)} sentences, "
          f"{total_truncated} sentences truncated at {MAX_SENTENCE_TOKENS} BPE tokens")
    return result

DOC_FEATURES = extract_document_features(DOCUMENTS)
del backbone, checkpoint
gc.collect()
if DEVICE.type == "cuda": torch.cuda.empty_cache()

In [ ]:
# Keep train and development documents separate. The incumbent was trained on
# train documents and selected on dev; the continuation below follows the same split.
TRAIN_DOCS = {
    domain: [DOC_FEATURES[f"{domain}::{str(row['id'])}"] for row in rows]
    for domain, rows in [("CL", CL_TRAIN), ("IT", IT_TRAIN)]}
DEV_DOCS = {
    domain: [DOC_FEATURES[f"{domain}::{str(row['id'])}"] for row in rows]
    for domain, rows in [("CL", CL_DEV), ("IT", IT_DEV)]}
for domain in ("CL", "IT"):
    train_keys = {row["key"] for row in TRAIN_DOCS[domain]}
    dev_keys = {row["key"] for row in DEV_DOCS[domain]}
    if train_keys & dev_keys:
        raise ValueError(f"{domain}: train/dev document overlap")
    print(domain, "training docs", len(train_keys), "development docs", len(dev_keys))


def collate_feature_docs(rows):
    features = pad_sequence([row["features_last"] for row in rows],
                            batch_first=True, padding_value=0.0)
    lengths = torch.tensor([len(row["labels"]) if row["labels"] is not None
                            else len(row["sentences"]) for row in rows], dtype=torch.long)
    labels = None
    if rows[0]["labels"] is not None:
        labels = pad_sequence([row["labels_tensor"] for row in rows],
                              batch_first=True, padding_value=-100)
    return features.to(DEVICE, non_blocking=True), lengths, (None if labels is None else labels.to(DEVICE, non_blocking=True))


class SequenceTagger(nn.Module):
    """V1-context BiLSTM head with state names mapped from the saved V3 model."""
    def __init__(self, n_labels=N_LABELS):
        super().__init__()
        self.context = nn.LSTM(768, 256, num_layers=2, batch_first=True,
                               bidirectional=True, dropout=0.25)
        self.dropout = nn.Dropout(0.25)
        self.classifier = nn.Linear(512, n_labels)

    def forward(self, features, lengths):
        packed = pack_padded_sequence(features, lengths.cpu(), batch_first=True,
                                      enforce_sorted=False)
        packed_out, _ = self.context(packed)
        sequence, _ = pad_packed_sequence(packed_out, batch_first=True,
                                          total_length=features.shape[1])
        return self.classifier(self.dropout(sequence))


## Restore the incumbent, then test one controlled continuation

The saved V3 checkpoint uses the same two-layer BiLSTM shape as the later sequence head, but its LSTM keys are named `lstm.*`. The loader maps those keys explicitly to `context.*` and then requires a strict state-dictionary match. This prevents silent partial loading.

The continuation starts from that exact state, uses one conservative learning rate, and trains only on train documents. Development data controls early stopping and the promotion guard. This is a small, targeted experiment rather than another architecture sweep.


In [ ]:
def evaluate_context_domain(model, rows):
    model.eval()
    gold, pred = [], []
    for start in range(0, len(rows), 2):
        batch = rows[start:start+2]
        features, lengths, labels = collate_feature_docs(batch)
        if labels is None:
            raise ValueError("Development documents must include labels")
        with torch.inference_mode():
            logits = model(features, lengths)
        for i, row in enumerate(batch):
            n = len(row["labels"])
            gold.extend(row["labels"])
            pred.extend(logits[i, :n].argmax(dim=-1).cpu().tolist())
    return metrics_from_ids(gold, pred)


def evaluate_context_model(model):
    return {domain: evaluate_context_domain(model, DEV_DOCS[domain])
            for domain in ("CL", "IT")}


def mean_domain_score(metrics):
    return (metrics["CL"]["macro_f1"] + metrics["IT"]["macro_f1"]) / 2.0


def paired_train_batches(epoch):
    cl_docs, it_docs = TRAIN_DOCS["CL"], TRAIN_DOCS["IT"]
    n = min(len(cl_docs), len(it_docs))
    if n == 0:
        raise ValueError("Both domains need training documents")
    rng = np.random.default_rng(SEED + 7100 + epoch)
    cl_idx = rng.choice(len(cl_docs), size=n, replace=False)
    it_idx = rng.choice(len(it_docs), size=n, replace=False)
    order = rng.permutation(n)
    for pos in order:
        rows = [cl_docs[int(cl_idx[pos])], it_docs[int(it_idx[pos])]]
        features, lengths, labels = collate_feature_docs(rows)
        yield features, lengths, labels


def weighted_pair_loss(logits, labels, lengths, class_weights):
    domain_losses = []
    for i in range(labels.shape[0]):
        n = int(lengths[i])
        targets = labels[i, :n]
        token_loss = F.cross_entropy(logits[i, :n].float(), targets,
                                     weight=class_weights, reduction="none")
        denominator = class_weights[targets].sum().clamp_min(1e-12)
        domain_losses.append(token_loss.sum() / denominator)
    return torch.stack(domain_losses).mean()


INCUMBENT_PATH = ASSETS["best_phase3_v1_context_bilstm.pt"]
INCUMBENT_CHECKPOINT = torch.load(INCUMBENT_PATH, map_location="cpu", weights_only=True)
if INCUMBENT_CHECKPOINT.get("label_to_id") != label_to_id:
    raise ValueError("Incumbent checkpoint label mapping differs from label_vocab.json")
for key in ("vocab_size", "n_embd", "n_layer", "n_head", "n_kv_head", "ffn_multiplier", "context_length"):
    if key in INCUMBENT_CHECKPOINT.get("v1_config", {}) and int(INCUMBENT_CHECKPOINT["v1_config"][key]) != int(model_config[key]):
        raise ValueError(f"Incumbent/V1 checkpoint mismatch for {key}")

INCUMBENT_MODEL = SequenceTagger().to(DEVICE)
old_state = INCUMBENT_CHECKPOINT["context_state_dict"]
converted_state = {}
for key, value in old_state.items():
    new_key = "context." + key[len("lstm."):] if key.startswith("lstm.") else key
    converted_state[new_key] = value
if set(converted_state) != set(INCUMBENT_MODEL.state_dict()):
    missing = sorted(set(INCUMBENT_MODEL.state_dict()) - set(converted_state))
    extra = sorted(set(converted_state) - set(INCUMBENT_MODEL.state_dict()))
    raise ValueError(f"Incumbent checkpoint keys do not match the model; missing={missing}, extra={extra}")
INCUMBENT_MODEL.load_state_dict(converted_state, strict=True)
INCUMBENT_MODEL.eval()

INCUMBENT_DEV_METRICS = evaluate_context_model(INCUMBENT_MODEL)
INCUMBENT_DEV_SCORE = mean_domain_score(INCUMBENT_DEV_METRICS)
RECORDED_DEV_SCORE = float(INCUMBENT_CHECKPOINT.get("selection_score", -1.0))
RECORDED_DEV_METRICS = INCUMBENT_CHECKPOINT.get("dev_metrics", {})
BASELINE_REPRODUCTION_TOL = 0.005
INCUMBENT_REPRODUCTION_OK = all(
    domain in RECORDED_DEV_METRICS
    and abs(INCUMBENT_DEV_METRICS[domain]["macro_f1"] -
            float(RECORDED_DEV_METRICS[domain]["macro_f1"])) <= BASELINE_REPRODUCTION_TOL
    for domain in ("CL", "IT"))
print("Recovered incumbent checkpoint:", INCUMBENT_PATH)
print("Stored incumbent dev score:", round(RECORDED_DEV_SCORE, 4))
print("Re-evaluated incumbent dev score:", round(INCUMBENT_DEV_SCORE, 4))
print("Incumbent feature/model reproduction check passed:", INCUMBENT_REPRODUCTION_OK)
if not INCUMBENT_REPRODUCTION_OK:
    print("WARNING: incumbent dev metrics did not reproduce closely; candidate promotion is disabled.")
for domain in ("CL", "IT"):
    print(domain, "incumbent fixed-13 dev Macro-F1:", round(INCUMBENT_DEV_METRICS[domain]["macro_f1"], 4))

# One low-rate fine-tuning run from the proven Phase 3 weights.
random.seed(SEED + 8801); np.random.seed(SEED + 8801); torch.manual_seed(SEED + 8801)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED + 8801)
CANDIDATE_MODEL = copy.deepcopy(INCUMBENT_MODEL).to(DEVICE)
checkpoint_weights = torch.tensor(INCUMBENT_CHECKPOINT["training_config"]["class_weights"],
                                 dtype=torch.float32, device=DEVICE)
if checkpoint_weights.numel() != N_LABELS or not torch.isfinite(checkpoint_weights).all():
    raise ValueError("Incumbent class weights are missing or invalid")
FINE_TUNE_LR = 5e-5
FINE_TUNE_MAX_EPOCHS = 4
FINE_TUNE_PATIENCE = 2
MIN_DEV_GAIN = 0.01
MAX_DOMAIN_DROP = 0.01
optimizer = torch.optim.AdamW(CANDIDATE_MODEL.parameters(), lr=FINE_TUNE_LR,
                              weight_decay=1e-3)
CANDIDATE_HISTORY = []
BEST_CANDIDATE_SCORE = INCUMBENT_DEV_SCORE
BEST_CANDIDATE_METRICS = INCUMBENT_DEV_METRICS
BEST_CANDIDATE_STATE = {k: v.detach().cpu().clone()
                        for k, v in CANDIDATE_MODEL.state_dict().items()}
BEST_CANDIDATE_EPOCH = 0
stale_epochs = 0

for epoch in range(1, FINE_TUNE_MAX_EPOCHS + 1):
    if not INCUMBENT_REPRODUCTION_OK:
        print("Skipping fine-tuning because the incumbent/dev reproduction check failed.")
        break
    CANDIDATE_MODEL.train()
    total_loss, steps = 0.0, 0
    for features, lengths, labels in paired_train_batches(epoch):
        optimizer.zero_grad(set_to_none=True)
        logits = CANDIDATE_MODEL(features, lengths)
        loss = weighted_pair_loss(logits, labels, lengths, checkpoint_weights)
        if not torch.isfinite(loss):
            raise FloatingPointError("Fine-tuning produced a non-finite loss")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(CANDIDATE_MODEL.parameters(), 1.0)
        optimizer.step()
        total_loss += float(loss.detach())
        steps += 1

    epoch_metrics = evaluate_context_model(CANDIDATE_MODEL)
    epoch_score = mean_domain_score(epoch_metrics)
    CANDIDATE_HISTORY.append({"epoch": epoch, "mean_pair_loss": total_loss / max(steps, 1),
                              "dev_mean_macro_f1": epoch_score,
                              "dev_CL_macro_f1": epoch_metrics["CL"]["macro_f1"],
                              "dev_IT_macro_f1": epoch_metrics["IT"]["macro_f1"]})
    print(f"Fine-tune epoch {epoch}/{FINE_TUNE_MAX_EPOCHS} | loss {total_loss/max(steps,1):.4f} | "
          f"CL {epoch_metrics['CL']['macro_f1']:.4f} | IT {epoch_metrics['IT']['macro_f1']:.4f} | "
          f"mean {epoch_score:.4f}")
    if epoch_score > BEST_CANDIDATE_SCORE:
        BEST_CANDIDATE_SCORE = epoch_score
        BEST_CANDIDATE_METRICS = epoch_metrics
        BEST_CANDIDATE_STATE = {k: v.detach().cpu().clone()
                                for k, v in CANDIDATE_MODEL.state_dict().items()}
        BEST_CANDIDATE_EPOCH = epoch
        stale_epochs = 0
    else:
        stale_epochs += 1
        if stale_epochs >= FINE_TUNE_PATIENCE:
            print("Fine-tuning stopped: development score did not improve.")
            break

CANDIDATE_MODEL.load_state_dict(BEST_CANDIDATE_STATE, strict=True)
CANDIDATE_ACCEPTED = (
    INCUMBENT_REPRODUCTION_OK
    and BEST_CANDIDATE_EPOCH > 0
    and BEST_CANDIDATE_SCORE >= INCUMBENT_DEV_SCORE + MIN_DEV_GAIN
    and all(BEST_CANDIDATE_METRICS[d]["macro_f1"] >=
            INCUMBENT_DEV_METRICS[d]["macro_f1"] - MAX_DOMAIN_DROP
            for d in ("CL", "IT")))
SELECTED_MODEL = CANDIDATE_MODEL if CANDIDATE_ACCEPTED else INCUMBENT_MODEL
SELECTED_KIND = "low_rate_continuation" if CANDIDATE_ACCEPTED else "incumbent_04311_fallback"
SELECTED_DEV_SCORE = BEST_CANDIDATE_SCORE if CANDIDATE_ACCEPTED else INCUMBENT_DEV_SCORE
print("Candidate best epoch:", BEST_CANDIDATE_EPOCH)
print("Candidate best dev score:", round(BEST_CANDIDATE_SCORE, 4))
print("Promotion guard passed:", CANDIDATE_ACCEPTED)
print("Selected model:", SELECTED_KIND)

with open(OUTPUT_DIR / "final_training_history.json", "w", encoding="utf-8") as f:
    json.dump(CANDIDATE_HISTORY, f, ensure_ascii=False, indent=2)
with open(OUTPUT_DIR / "incumbent_dev_metrics.json", "w", encoding="utf-8") as f:
    json.dump({"checkpoint_recorded_score": RECORDED_DEV_SCORE,
               "re_evaluated_mean_domain_macro_f1": INCUMBENT_DEV_SCORE,
               "domains": INCUMBENT_DEV_METRICS}, f, ensure_ascii=False, indent=2)
with open(OUTPUT_DIR / "candidate_dev_metrics.json", "w", encoding="utf-8") as f:
    json.dump({"best_epoch": BEST_CANDIDATE_EPOCH,
               "best_mean_domain_macro_f1": BEST_CANDIDATE_SCORE,
               "domains": BEST_CANDIDATE_METRICS,
               "promotion_guard_passed": CANDIDATE_ACCEPTED}, f, ensure_ascii=False, indent=2)


## Safe selection and Codabench export

The notebook always prepares the candidate predictions for review. It promotes them only after the stated development guard passes. If not, `prediction.json` is built from the exact saved 0.4311-run prediction file, preserving the previous best output byte-for-byte at the JSON content level.

Before creating the ZIP, the notebook validates test IDs, document order, sentence counts, official label strings, the one-file archive, and ZIP CRC.


In [ ]:
def predict_test_documents(model, domain):
    model.eval()
    source_docs = CL_TEST if domain == "CL" else IT_TEST
    output = []
    for doc_id, sentences in source_docs:
        row = DOC_FEATURES[f"{domain}::{str(doc_id)}"]
        features = row["features_last"].unsqueeze(0).to(DEVICE)
        lengths = torch.tensor([len(sentences)], dtype=torch.long)
        with torch.inference_mode():
            logits = model(features, lengths)
        ids = logits[0, :len(sentences)].argmax(dim=-1).cpu().tolist()
        labels = [id_to_label[int(i)] for i in ids]
        if len(labels) != len(sentences) or not set(labels) <= label_set:
            raise ValueError(f"{domain}/{doc_id}: invalid test predictions")
        output.append({"id": str(doc_id), "predictions": labels})
    return output


def rows_to_flat(rows_by_domain):
    flat = {}
    for domain in ("CL", "IT"):
        for row in rows_by_domain[domain]:
            key = str(row["id"])
            if key in flat:
                raise ValueError(f"Duplicate prediction document ID: {key}")
            flat[key] = list(row["predictions"])
    return flat

expected_lengths = {}
for docs in (CL_TEST, IT_TEST):
    for doc_id, sentences in docs:
        key = str(doc_id)
        if key in expected_lengths:
            raise ValueError(f"Duplicate test document ID: {key}")
        expected_lengths[key] = len(sentences)

with open(ASSETS["best_04311_prediction.json"], "r", encoding="utf-8") as f:
    INCUMBENT_REFERENCE = json.load(f)
if not isinstance(INCUMBENT_REFERENCE, dict):
    raise TypeError("Incumbent reference must be a flat document-ID JSON object")
if set(INCUMBENT_REFERENCE) != set(expected_lengths):
    raise ValueError("Incumbent reference document IDs do not match the attached test data")
for doc_id, labels in INCUMBENT_REFERENCE.items():
    if not isinstance(labels, list) or len(labels) != expected_lengths[doc_id]:
        raise ValueError(f"Incumbent reference has an invalid sentence count for {doc_id}")
    if any(not isinstance(label, str) for label in labels) or not set(labels) <= label_set:
        raise ValueError(f"Incumbent reference has an unknown label for {doc_id}")

# Recompute incumbent and candidate predictions for diagnosis. If the candidate
# fails the dev guard, the exact saved incumbent JSON remains the submission.
INCUMBENT_TEST_ROWS = {
    "CL": predict_test_documents(INCUMBENT_MODEL, "CL"),
    "IT": predict_test_documents(INCUMBENT_MODEL, "IT")}
CANDIDATE_TEST_ROWS = {
    "CL": predict_test_documents(CANDIDATE_MODEL, "CL"),
    "IT": predict_test_documents(CANDIDATE_MODEL, "IT")}
INCUMBENT_RECOMPUTED = rows_to_flat(INCUMBENT_TEST_ROWS)
INCUMBENT_REFERENCE_MATCH = INCUMBENT_RECOMPUTED == INCUMBENT_REFERENCE
mismatch_count = sum(
    INCUMBENT_RECOMPUTED[doc_id][i] != INCUMBENT_REFERENCE[doc_id][i]
    for doc_id in expected_lengths
    for i in range(expected_lengths[doc_id]))
print("Recomputed incumbent exactly matches saved prediction file:", INCUMBENT_REFERENCE_MATCH)
print("Incumbent label mismatches, if any:", mismatch_count)

if CANDIDATE_ACCEPTED:
    SELECTED_ROWS = CANDIDATE_TEST_ROWS
    SELECTED_PREDICTIONS = rows_to_flat(SELECTED_ROWS)
else:
    SELECTED_PREDICTIONS = INCUMBENT_REFERENCE
    SELECTED_ROWS = {
        domain: [{"id": str(doc_id), "predictions": list(INCUMBENT_REFERENCE[str(doc_id)])}
                 for doc_id, _ in (CL_TEST if domain == "CL" else IT_TEST)]
        for domain in ("CL", "IT")}


def validate_submission(payload, source):
    if not isinstance(payload, dict):
        raise TypeError(f"{source}: expected flat object keyed by document ID")
    if set(payload) != set(expected_lengths):
        if set(payload) == {"CL", "IT"}:
            raise ValueError(f"{source}: found domain-wrapped JSON; expected flat document IDs")
        raise ValueError(f"{source}: IDs differ from the attached test data")
    for doc_id, roles in payload.items():
        if not isinstance(roles, list) or len(roles) != expected_lengths[doc_id]:
            raise ValueError(f"{source}/{doc_id}: sentence count mismatch")
        if any(not isinstance(role, str) for role in roles) or not set(roles) <= label_set:
            raise ValueError(f"{source}/{doc_id}: unknown or non-string label")

validate_submission(SELECTED_PREDICTIONS, "selected predictions")
for domain in ("CL", "IT"):
    with open(OUTPUT_DIR / f"{domain}_incumbent_recomputed.json", "w", encoding="utf-8") as f:
        json.dump({"domain": domain, "documents": INCUMBENT_TEST_ROWS[domain]}, f, ensure_ascii=False, indent=2)
    with open(OUTPUT_DIR / f"{domain}_candidate_predictions.json", "w", encoding="utf-8") as f:
        json.dump({"domain": domain, "documents": CANDIDATE_TEST_ROWS[domain]}, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "phase3_rr_predictions_by_domain.json", "w", encoding="utf-8") as f:
    json.dump(SELECTED_ROWS, f, ensure_ascii=False, indent=2)
PREDICTION_PATH = OUTPUT_DIR / "prediction.json"
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(SELECTED_PREDICTIONS, f, ensure_ascii=False, indent=2)
with open(PREDICTION_PATH, "r", encoding="utf-8") as f:
    serialized = json.load(f)
validate_submission(serialized, "prediction.json")
if serialized != SELECTED_PREDICTIONS:
    raise ValueError("Saved prediction.json differs from the validated in-memory output")

ZIP_PATH = OUTPUT_DIR / "submission.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(PREDICTION_PATH, arcname="prediction.json")
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    if archive.namelist() != ["prediction.json"]:
        raise ValueError(f"Unexpected ZIP members: {archive.namelist()}")
    if archive.testzip() is not None:
        raise ValueError("Submission ZIP CRC validation failed")
    packaged = json.loads(archive.read("prediction.json").decode("utf-8"))
validate_submission(packaged, "submission.zip/prediction.json")
if packaged != serialized:
    raise ValueError("ZIP payload does not exactly match prediction.json")

selected_state = SELECTED_MODEL.state_dict()
selected_checkpoint_path = OUTPUT_DIR / "selected_phase3_context_model.pt"
torch.save({"model_state_dict": selected_state,
            "selected_model": SELECTED_KIND,
            "label_to_id": label_to_id,
            "v1_config": model_config,
            "incumbent_dev_score": INCUMBENT_DEV_SCORE,
            "candidate_dev_score": BEST_CANDIDATE_SCORE,
            "candidate_accepted": CANDIDATE_ACCEPTED}, selected_checkpoint_path)
comparison = {
    "selected_model": SELECTED_KIND,
    "selected_dev_mean_domain_macro_f1": SELECTED_DEV_SCORE,
    "incumbent_reported_codabench_score": 0.4311,
    "incumbent_dev_mean_domain_macro_f1": INCUMBENT_DEV_SCORE,
    "incumbent_checkpoint_recorded_dev_score": RECORDED_DEV_SCORE,
    "incumbent_reproduction_check_passed": INCUMBENT_REPRODUCTION_OK,
    "incumbent_reproduction_tolerance_per_domain": BASELINE_REPRODUCTION_TOL,
    "candidate_best_dev_mean_domain_macro_f1": BEST_CANDIDATE_SCORE,
    "candidate_best_epoch": BEST_CANDIDATE_EPOCH,
    "candidate_accepted": CANDIDATE_ACCEPTED,
    "candidate_rule": {"minimum_mean_dev_gain": MIN_DEV_GAIN,
                        "maximum_allowed_drop_per_domain": MAX_DOMAIN_DROP},
    "incumbent_test_recomputation_matches_saved_json": INCUMBENT_REFERENCE_MATCH,
    "incumbent_recomputed_label_mismatches": mismatch_count,
    "fallback_prediction_source": str(ASSETS["best_04311_prediction.json"]),
    "phase1_v1_retrained": False,
    "phase3_training": "low-rate continuation from saved context BiLSTM; train split only",
    "submission_documents": len(serialized),
    "prediction_file": str(PREDICTION_PATH),
    "submission_zip": str(ZIP_PATH),
    "selection_note": "The saved 0.4311 prediction JSON is used unless the single continuation passes the conservative development guard. This does not guarantee a higher hidden-test score."}
with open(OUTPUT_DIR / "model_comparison.json", "w", encoding="utf-8") as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)
print("Selected model:", SELECTED_KIND)
print("Submission JSON:", PREDICTION_PATH)
print("Submission ZIP:", ZIP_PATH)
print("PASS: exact test IDs, sentence counts, labels, single root prediction.json, and ZIP CRC validated.")


## How to interpret the result

A falling training loss is not the promotion criterion. The incumbent and its development metrics are recorded, each fine-tuning epoch is logged, and the candidate must clear the conservative guard before it can replace the saved 0.4311 prediction file. The hidden test labels remain unavailable, so no notebook can promise that a candidate will score higher.

The output ZIP is always structurally checked. If the candidate is not accepted, the ZIP contains the known incumbent prediction payload. If the candidate is accepted, the ZIP contains the new model predictions; submit it only after reviewing `model_comparison.json` and the domain metrics.
